# 第 10 章 労働市場と AD-AS 分析: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-basics-programming/10-labor-market-and-ad-as.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
demand_intercept = 100  # 労働需要 Ld = a − b·w の切片 a
demand_slope = 10       # 労働需要の傾き b
supply_intercept = 20   # 労働供給 Ls = c + d·w の切片 c
supply_slope = 10       # 労働供給の傾き d
sticky_wage = 6         # 硬直的な実質賃金
subsistence = 1         # 効率賃金仮説の、生存水準の賃金
mpc = 0.6               # 限界消費性向 c
b = 20                  # 投資の利子感応度 b
intercept = 450         # IS 曲線の A
k = 0.5                 # 貨幣需要の所得感応度 k
h = 100                 # 貨幣需要の利子感応度 h
nominal_money = 250     # 名目貨幣供給 M
full_employment_income = 1000  # 完全雇用所得
base_income = 480       # ケインズの AS の、基準の所得
slope = 400             # ケインズの AS の傾き
delta_government = 10   # 政府支出の増加（ディマンドプル）
supply_loss = 50        # 供給の減少（コストプッシュ）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_loaded_lib = sys.modules.get("lib")
if _loaded_lib is not None and (
    Path(getattr(_loaded_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in tuple(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 労働市場

In [ ]:
from lib.chapter10.labor import (
    LaborCurves,
    best_efficiency_wage,
    involuntary_unemployment,
    labor_equilibrium,
)

curves = LaborCurves(demand_intercept, demand_slope, supply_intercept, supply_slope)
wage, employment = labor_equilibrium(curves)
print(f"均衡: 実質賃金 {wage:.2f}、雇用 {employment:.1f}")
print(f"実質賃金が {sticky_wage} に硬直的なときの、非自発的失業: "
      f"{involuntary_unemployment(curves, sticky_wage):.1f}")
print("効率賃金（努力 ÷ 賃金が最大）:",
      best_efficiency_wage([2, 3, 4, 5, 6, 8, 10], subsistence))

## 図: 労働市場

In [ ]:
from lib.chapter10.plot import plot_labor_market

plot_labor_market(curves, sticky_wage, wage_max=10)

## AD と AS の交点

In [ ]:
from functools import partial

from lib.chapter10.ad_as import (
    AdEconomy,
    expand_demand,
    keynesian_supply,
    shift_supply,
    solve_ad_as,
)

economy = AdEconomy(mpc, b, intercept, k, h, nominal_money)
supply = partial(
    keynesian_supply,
    full_employment_income=full_employment_income,
    base_income=base_income,
    slope=slope,
)
price, income = solve_ad_as(economy, supply)
print(f"もとの均衡: 物価 {price:.3f}、所得 {income:.1f}")
price1, income1 = solve_ad_as(expand_demand(economy, delta_government), supply)
print(f"AD が右へ: 物価 {price1:.3f}、所得 {income1:.1f}")
full, base_after = shift_supply(full_employment_income, base_income, -supply_loss)
shifted = partial(
    keynesian_supply, full_employment_income=full, base_income=base_after, slope=slope
)
price2, income2 = solve_ad_as(economy, shifted)
print(f"AS が左へ: 物価 {price2:.3f}、所得 {income2:.1f}")

## 図: AD-AS

In [ ]:
from lib.chapter10.plot import plot_ad_as

plot_ad_as(
    economy, full_employment_income, base_income, slope, delta_government,
    price_min=0.8, price_max=1.6,
)

## 試してみよう

- `sticky_wage = 7` にすると、非自発的失業は 60.0 になります（6 のときは 40.0）。
  `sticky_wage = 4` にすると、均衡（4.0）なので、失業は 0.0 です
- `subsistence = 2` にすると、効率賃金は 4 になります（1 のときは 2）。
  生存水準が高いほど、企業は高い賃金を払うのが得になります
- もとの均衡は、物価 1.250、所得 980.0 です。`delta_government = 10`（AD が右へ）で、
  物価 1.293、所得 997.3 になります。**物価と所得が、ともに上がります**
- `supply_loss = 50`（AS が左へ）では、物価 2.000、所得 950.0 です。**物価が上がり、
  所得が下がります**（スタグフレーション）